In [3]:
!pip install -q jiwer

In [4]:
import os
import random
import numpy as np
import tensorflow as tf
import tensorflow_datasets as tfds
import matplotlib.pyplot as plt
from tensorflow import keras
from tensorflow.keras import layers
from jiwer import wer

# ==========================================
# 1. КОНФІГУРАЦІЯ ТА НАЛАШТУВАННЯ
# ==========================================
class ProjectConfig:
    # Параметри аудіо
    FRAME_LENGTH = 256
    FRAME_STEP = 256
    FFT_SIZE = 256
    
    # Параметри навчання
    BATCH_SIZE = 32
    EPOCHS = 50 
    PATIENCE = 10
    LEARNING_RATE = 1e-4
    
    # Шляхи
    MODEL_FILE = "deepspeech2_ljspeech.keras"
    
    # Словник символів (англійська)
    VOCAB_STR = "abcdefghijklmnopqrstuvwxyz'?! "

config = ProjectConfig()

# ==========================================
# 2. ПІДГОТОВКА ДАНИХ (ETL PIPELINE)
# ==========================================

# 2.1 Створення шарів для кодування тексту
char_to_num = layers.StringLookup(vocabulary=list(config.VOCAB_STR), oov_token="")
num_to_char = layers.StringLookup(
    vocabulary=char_to_num.get_vocabulary(), oov_token="", invert=True
)

# 2.2 Функція обробки: Аудіо -> Спектрограма
def process_audio_sample(sample):
    # Отримуємо аудіо та текст
    audio = sample["speech"]
    audio = tf.cast(audio, tf.float32)
    
    # Створення спектрограми (Short-time Fourier transform)
    spectrogram = tf.signal.stft(
        audio, 
        frame_length=config.FRAME_LENGTH, 
        frame_step=config.FRAME_STEP, 
        fft_length=config.FFT_SIZE
    )
    
    # Перетворення в амплітуду та нормалізація динамічного діапазону
    spectrogram = tf.abs(spectrogram)
    spectrogram = tf.math.pow(spectrogram, 0.5)
    
    # Стандартизація (mean=0, std=1)
    means = tf.math.reduce_mean(spectrogram, 1, keepdims=True)
    stddevs = tf.math.reduce_std(spectrogram, 1, keepdims=True)
    spectrogram = (spectrogram - means) / (stddevs + 1e-10)
    
    # Обробка тексту (лейблу)
    label = tf.strings.lower(sample["text"])
    label = tf.strings.unicode_split(label, input_encoding="UTF-8")
    label = char_to_num(label)
    
    return spectrogram, label

# 2.3 Завантаження та формування пайплайну
print("Завантаження датасету LJSpeech ...")
dataset = tfds.load("ljspeech", split="train", as_supervised=False)

# Розділення на train/val
validation_split = dataset.take(200) # Беремо трохи більше для валідації
training_split = dataset.skip(200)

def create_dataset_pipeline(ds, bs):
    return (
        ds.map(process_audio_sample, num_parallel_calls=tf.data.AUTOTUNE)
        .cache() # Кешування в RAM
        .shuffle(1000)
        .bucket_by_sequence_length(
            element_length_func=lambda spec, label: tf.shape(spec)[0],
            bucket_boundaries=[200, 300, 400, 500, 600, 700, 800],
            bucket_batch_sizes=[bs] * 8,
            pad_to_bucket_boundary=False
        )
        .prefetch(tf.data.AUTOTUNE)
    )

train_dataset = create_dataset_pipeline(training_split, config.BATCH_SIZE)
val_dataset = create_dataset_pipeline(validation_split, config.BATCH_SIZE)

# ==========================================
# 4. АРХІТЕКТУРА НЕЙРОННОЇ МЕРЕЖІ (DS2)
# ==========================================

# Функція втрат CTC
def ctc_loss(y_true, y_pred):
    batch_len = tf.cast(tf.shape(y_true)[0], dtype="int64")
    input_length = tf.cast(tf.shape(y_pred)[1], dtype="int64")
    label_length = tf.cast(tf.shape(y_true)[1], dtype="int64")

    input_length = input_length * tf.ones(shape=(batch_len, 1), dtype="int64")
    label_length = label_length * tf.ones(shape=(batch_len, 1), dtype="int64")

    return keras.backend.ctc_batch_cost(y_true, y_pred, input_length, label_length)

def build_deepspeech_model(input_dim, output_dim, rnn_units=256):
    input_spectrogram = layers.Input((None, input_dim), name="input_spec")
    
    # Розширення розмірності для CNN [Batch, Time, Freq, Channel]
    x = layers.Reshape((-1, input_dim, 1), name="expand_dim")(input_spectrogram)
    
    # Блок згорток (CNN) для виділення локальних ознак
    x = layers.Conv2D(32, (11, 41), strides=(2, 2), padding="same", activation="relu", name="conv_1")(x)
    x = layers.BatchNormalization(name="bn_1")(x)
    x = layers.Conv2D(32, (11, 21), strides=(1, 2), padding="same", activation="relu", name="conv_2")(x)
    x = layers.BatchNormalization(name="bn_2")(x)
    
    # Підготовка до рекурентних шарів
    # Вираховуємо нову розмірність features після згортки
    new_shape = (-1, x.shape[-2] * x.shape[-1]) 
    x = layers.Reshape(new_shape, name="flatten_features")(x)
    
    x = layers.Dense(rnn_units, activation="relu", name="dense_proj")(x)
    x = layers.Dropout(0.2)(x)
    
    # Блок рекурентних шарів (Bidirectional LSTM)
    x = layers.Bidirectional(layers.LSTM(rnn_units, return_sequences=True), name="rnn_1")(x)
    x = layers.Dropout(0.2)(x)
    x = layers.Bidirectional(layers.LSTM(rnn_units, return_sequences=True), name="rnn_2")(x)
    x = layers.Dropout(0.2)(x)
    
    # Вихідний шар (Softmax над словником + blank token для CTC)
    output = layers.Dense(output_dim + 1, activation="softmax", name="logits")(x)
    
    model = keras.Model(input_spectrogram, output, name="DeepSpeech_2_Lite")
    optimizer = keras.optimizers.Adam(learning_rate=config.LEARNING_RATE)
    
    model.compile(optimizer=optimizer, loss=ctc_loss)
    return model

# Побудова моделі
input_freq_bins = config.FFT_SIZE // 2 + 1
vocab_size = char_to_num.vocabulary_size()

model = build_deepspeech_model(input_freq_bins, vocab_size)
model.summary()

# ==========================================
# 5. НАВЧАННЯ
# ==========================================

# Callbacks
callbacks = [
    keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=config.PATIENCE, restore_best_weights=True, verbose=1
    ),
    keras.callbacks.ModelCheckpoint(
        config.MODEL_FILE, monitor="val_loss", save_best_only=True
    )
]

print("\nПочинаємо навчання...")
history = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=config.EPOCHS,
    callbacks=callbacks
)
print("Навчання завершено.")

Завантаження датасету LJSpeech ...


Model: "DeepSpeech_2_Lite"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_spec (InputLayer)         │ (None, None, 129)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ expand_dim (Reshape)            │ (None, None, 129, 1)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_1 (Conv2D)                 │ (None, None, 65, 32)   │        14,464 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bn_1 (BatchNormalization)       │ (None, None, 65, 32)   │           128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_2 (Conv2D)                 │ (None, None, 33, 32)   │       236,576 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bn_2 (BatchNormalization)       │ (None, None, 33, 32)   │           128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_features (Reshape)      │ (None, None, 1056)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_proj (Dense)              │ (None, None, 256)      │       270,592 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, None, 256)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ rnn_1 (Bidirectional)           │ (None, None, 512)      │     1,050,624 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, None, 512)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ rnn_2 (Bidirectional)           │ (None, None, 512)      │     1,574,912 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, None, 512)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ logits (Dense)                  │ (None, None, 32)       │        16,416 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,163,840 (12.07 MB)

 Trainable params: 3,163,712 (12.07 MB)

 Non-trainable params: 128 (512.00 B)


Починаємо навчання...
Epoch 1/50
408/408 ━━━━━━━━━━━━━━━━━━━━ 141s 326ms/step - loss: 345.1573 - val_loss: 296.2200
Epoch 2/50
408/408 ━━━━━━━━━━━━━━━━━━━━ 119s 292ms/step - loss: 303.9415 - val_loss: 294.4046
Epoch 3/50
408/408 ━━━━━━━━━━━━━━━━━━━━ 118s 289ms/step - loss: 296.6417 - val_loss: 288.5161
Epoch 4/50
408/408 ━━━━━━━━━━━━━━━━━━━━ 113s 277ms/step - loss: 288.0449 - val_loss: 265.1664
Epoch 5/50
408/408 ━━━━━━━━━━━━━━━━━━━━ 115s 281ms/step - loss: 268.1750 - val_loss: 248.6442
Epoch 6/50
408/408 ━━━━━━━━━━━━━━━━━━━━ 114s 279ms/step - loss: 240.1922 - val_loss: 220.8218
Epoch 7/50
408/408 ━━━━━━━━━━━━━━━━━━━━ 110s 270ms/step - loss: 217.3834 - val_loss: 200.4625
Epoch 8/50
408/408 ━━━━━━━━━━━━━━━━━━━━ 106s 260ms/step - loss: 205.2564 - val_loss: 185.3998
Epoch 9/50
408/408 ━━━━━━━━━━━━━━━━━━━━ 107s 261ms/step - loss: 191.3421 - val_loss: 166.3883
Epoch 10/50
408/408 ━━━━━━━━━━━━━━━━━━━━ 110s 271ms/step - loss: 179.9440 - val_loss: 160.7384
Epoch 11/50
408/408 ━━━━━━━━━━━━━━━━

In [6]:
# ==========================================
# 6. ТЕСТУВАННЯ ТА ОЦІНКА (INFERENCE)
# ==========================================

def decode_batch_predictions(pred):
    input_len = np.ones(pred.shape[0]) * pred.shape[1]
    # Greedy Search decoding
    results = keras.backend.ctc_decode(pred, input_length=input_len, greedy=True)[0][0]
    
    output_text = []
    for result in results:
        res = tf.strings.reduce_join(num_to_char(result)).numpy().decode("utf-8")
        output_text.append(res)
    return output_text

# Збір статистики
predictions_data = []
total_wer = 0
count = 0
MAX_TEST_SAMPLES = 100  # Обмежимо кількість для швидкого тесту

print(f"\nТестування на {MAX_TEST_SAMPLES} семплах...")

for batch in val_dataset:
    if count >= MAX_TEST_SAMPLES:
        break
        
    X, y = batch
    batch_predictions = model.predict(X, verbose=0)
    batch_text_preds = decode_batch_predictions(batch_predictions)
    
    for i in range(len(batch_text_preds)):
        if count >= MAX_TEST_SAMPLES:
            break
            
        target_text = tf.strings.reduce_join(num_to_char(y[i])).numpy().decode("utf-8")
        pred_text = batch_text_preds[i]
        
        # Розрахунок помилки для конкретного речення
        current_wer = wer(target_text, pred_text)
        
        predictions_data.append({
            "target": target_text,
            "pred": pred_text,
            "wer": current_wer
        })
        
        total_wer += current_wer
        count += 1

# ==========================================
# 7. ЗВІТ ПО РЕЗУЛЬТАТАХ
# ==========================================
avg_wer = total_wer / count
predictions_data.sort(key=lambda x: x["wer"]) # Сортування від кращого до гіршого

print("\n" + "="*50)
print(f"ПІДСУМКОВИЙ ЗВІТ ТЕСТУВАННЯ (Samples: {count})")
print(f"СЕРЕДНІЙ WER: {avg_wer:.4f}")
print("="*50)

def print_cases(title, cases):
    print(f"\n--- {title} ---")
    for item in cases:
        print(f"WER: {item['wer']:.4f}")
        print(f"Оригінал:   {item['target']}")
        print(f"Розпізнано: {item['pred']}")
        print("-" * 20)

# Виводимо найкращі, найгірші та випадкові результати
print_cases("3 Найкращі результати", predictions_data[:3])
print_cases("3 Найгірші результати", predictions_data[-3:])

# Випадкові приклади для об'єктивності
print_cases("3 Випадкові приклади", random.sample(predictions_data, 3))


Тестування на 100 семплах...

ПІДСУМКОВИЙ ЗВІТ ТЕСТУВАННЯ (Samples: 100)
СЕРЕДНІЙ WER: 0.6159

--- 3 Найкращі результати ---
WER: 0.3000
Оригінал:   the reforms which were to be attempted in that prison
Розпізнано: the refoms which wer to be attempted in that prisoon
--------------------
WER: 0.3684
Оригінал:   our investigation of oswald had disclosed no evidence that oswald was acting under the instructions or on behalf of
Розпізнано: or investigationof oswald phad disclosed no evidence that oswald was acting ounder the instructions or on me haf
--------------------
WER: 0.3750
Оригінал:   it was also claimed for the more ample and more orderly distribution of victuals that the general health of the prisoners had greatly improved
Розпізнано: it was also cplaimed for the more ample and more morderly distrvution ofvituals that the general helvte of the prisoners and greately mprove
--------------------

--- 3 Найгірші результати ---
WER: 1.0000
Оригінал:   one heaping cup of sifted fl